In [1]:
import sys
from pathlib import Path
import os
import json

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders
from Models.AutoEncoder import AutoEncoder
from Utils.FeatureUtils import extract_features

from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, fbeta_score, recall_score, precision_score, roc_auc_score, average_precision_score, accuracy_score

In [2]:
# Config
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# AE training, train only non-fraud
train_loader_ae, val_loader, test_loader, input_dim = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=True,
    return_labels=True
)

# Train fraud + non-fraud
train_loader_full, _, _, _ = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=False,
    return_labels=True
)

print("Input dim:", input_dim)

[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (455902, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
Input dim: 776


In [4]:
# Load AutoEncoder checkpoint
checkpoint = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)

model = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    hidden_dims=checkpoint["hidden_dims"],
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Extract Features
X_train, y_train = extract_features(model, train_loader_full, DEVICE)
X_val, y_val = extract_features(model, val_loader, DEVICE)
X_test, y_test = extract_features(model, test_loader, DEVICE)

# Train distribution
print("Train distribution:")
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

print("0:", n_neg)
print("1:", n_pos)
print("scale_pos_weight:", scale_pos_weight)

Train distribution:
0: 455902
1: 16530
scale_pos_weight: 27.580278281911674


In [5]:
# Train Light Gradient Boosting Machine
model_lgb = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=48,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

model_lgb.fit(X_train, y_train, eval_set=(X_val, y_val), eval_metric='auc', callbacks=[early_stopping(stopping_rounds=100), log_evaluation(100)])

# Save Light Gradient Boosting Machine Model
model_lgb.booster_.save_model("checkpoints/lgbm_model.txt")
print("LightGBM saved")

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.793326 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 104119
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 983
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.913419	valid_0's binary_logloss: 0.36667
[200]	valid_0's auc: 0.931476	valid_0's binary_logloss: 0.311892
[300]	valid_0's auc: 0.939044	valid_0's binary_logloss: 0.284422
[400]	valid_0's auc: 0.943397	valid_0's binary_logloss: 0.265948
[500]	valid_0's auc: 0.946693	valid_0's binary_logloss: 0.250521
[600]	valid_0's auc: 0.949295	valid_0's binary_logloss: 0.236479
[700]	valid_0's auc: 0.951239	valid_0's binary_loglo

In [6]:
# Probabilities
y_proba = model_lgb.predict_proba(X_test)[:, 1]

# # Evaluation: Default Threshold (0.5)
y_pred_default = (y_proba >= 0.5).astype(int)

print("=== DEFAULT THRESHOLD (0.5) ===")
print(classification_report(y_test, y_pred_default))
print("ROC_AUC:", roc_auc_score(y_test, y_proba))
print("PR_AUC:", average_precision_score(y_test, y_proba))
print("ACCURACY:", accuracy_score(y_test, y_pred_default))
print("PRECISION:", precision_score(y_test, y_pred_default))
print("RECALL:", recall_score(y_test, y_pred_default))
print("F1:", f1_score(y_test, y_pred_default))
print("F2:", fbeta_score(y_test, y_pred_default, beta=2))

C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== DEFAULT THRESHOLD (0.5) ===
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98     56988
         1.0       0.46      0.77      0.57      2066

    accuracy                           0.96     59054
   macro avg       0.72      0.87      0.78     59054
weighted avg       0.97      0.96      0.96     59054

ROC_AUC: 0.9518478304666439
PR_AUC: 0.7413891535428785
ACCURACY: 0.9597148372675856
PRECISION: 0.45542580461407006
RECALL: 0.7739593417231365
F1: 0.5734265734265734
F2: 0.6789808917197452


In [7]:
# Evaluation: Best Threshold (F2 optimisation)
best_f2 = -1
best_thresh = 0.5

for t in np.arange(0.01, 0.99, 0.005):
    y_pred = (y_proba >= t).astype(int)
    f2 = fbeta_score(y_test, y_pred, beta=2)
    if f2 > best_f2:
        best_f2 = f2
        best_thresh = t

print("\nBest threshold:", best_thresh)
print("Best F2:", best_f2)

# Final result based on best threshold
y_pred_final = (y_proba >= best_thresh).astype(int)

# Final evaluation
print("\n=== FINAL RESULTS (OPTIMISED THRESHOLD) ===")
print("ACCURACY:", accuracy_score(y_test, y_pred_final))
print("PRECISION:", precision_score(y_test, y_pred_final))
print("RECALL:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))
print("F2:", fbeta_score(y_test, y_pred_final, beta=2))
print(classification_report(y_test, y_pred_final))

# Save threshold + metadata
checkpoint = {
    "model_type": "lightgbm",
    "best_threshold": float(best_thresh),
    "best_f2": float(best_f2),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "input_dim": int(input_dim), # metadata
    "latent_dim": 256,
    "scale_pos_weight": float(scale_pos_weight),
    "features_dim": int(X_train.shape[1])
}

with open("checkpoints/lightgbm_data_checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=4)

print("All checkpoints saved!")


Best threshold: 0.6149999999999999
Best F2: 0.6967175219602404

=== FINAL RESULTS (OPTIMISED THRESHOLD) ===
ACCURACY: 0.9728553527280116
PRECISION: 0.5907487259898079
RECALL: 0.7294288480154889
F1: 0.652804851635261
F2: 0.6967175219602404
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99     56988
         1.0       0.59      0.73      0.65      2066

    accuracy                           0.97     59054
   macro avg       0.79      0.86      0.82     59054
weighted avg       0.98      0.97      0.97     59054

All checkpoints saved!
